# 05 – Geo Lookup Preparation

**Purpose:** Build a fast nearest‑neighbour lookup for the live map click in the FastAPI backend.

**Why:**  
When a farmer clicks on the map, we get latitude and longitude. We need to return the closest GEE grid point’s rainfall and soil pH. This notebook creates a KDTree index and saves it for the backend.

**Output:** `backend/geo_lookup.pkl` containing:
- KDTree for fast spatial queries
- Corresponding coordinates
- Precomputed rainfall and pH arrays

In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy.spatial import cKDTree
import os

# Ensure backend directory exists
os.makedirs('../backend', exist_ok=True)

In [2]:
geo = pd.read_csv('../data/india_geo_features_complete.csv')
print(f"Loaded {len(geo)} GEE points.")
print(geo.head(15))

Loaded 1137 GEE points.
    longitude   latitude  avg_rainfall_mm  soil_ph   Nitrogen  Phosphorous  \
0   77.322488   8.152211      2698.178401      5.6  17.916667    17.920290   
1   76.828415   8.646285      4377.944946      5.7   0.057692    47.096154   
2   77.322488   8.646285      3421.822819      5.4   9.190741     9.190741   
3   77.816561   8.646285      2096.451408      7.2  15.160256    15.160256   
4   76.828415   9.140358      5433.940539      5.4   0.010256    32.348718   
5   77.322488   9.140358      3436.787036      5.4   9.190741     9.190741   
6   77.816561   9.140358      2070.078375      7.5  15.160256    15.160256   
7   78.310635   9.140358      1543.255322      7.2  17.349727    17.355972   
8   76.334341   9.634431      7153.444546      5.7   0.084291    28.842912   
9   76.828415   9.634431      5292.179458      5.1   0.041237    25.302405   
10  77.322488   9.634431      3848.868977      5.4  22.764706    22.764706   
11  77.816561   9.634431      2932.45660

In [5]:
coords = geo[['longitude', 'latitude']].values
values_climate = geo[['avg_rainfall_mm', 'soil_ph']].values
values_NPK = geo[['Nitrogen', 'Phosphorous', 'Potassium']].values

print(f"Coordinates shape: {coords.shape}")
print(f"Climate values shape: {values_climate.shape}")
print(f"NPK values shape: {values_NPK.shape}")

Coordinates shape: (1137, 2)
Climate values shape: (1137, 2)
NPK values shape: (1137, 3)


In [6]:
tree = cKDTree(coords)
print("KDTree built successfully.")

KDTree built successfully.


In [7]:
# Test with a sample coordinate (e.g., Delhi)
test_lon, test_lat = 72.1025, 28.7041
dist, idx = tree.query([[test_lon, test_lat]])
print(f"Closest point index: {idx[0]}")
print(f"Closest coordinates: {coords[idx[0]]}")
print(f"Rainfall: {values_climate[idx[0]][0]:.2f} mm/year")
print(f"Soil pH: {values_climate[idx[0]][1]:.2f}")
print(f"Nitrogen: {values_NPK[idx[0]][0]:.2f} mg/kg")
print(f"Phosphorous: {values_NPK[idx[0]][1]:.2f} mg/kg")
print(f"Potassium: {values_NPK[idx[0]][2]:.2f} mg/kg")

Closest point index: 975
Closest coordinates: [72.38175402 28.40922086]
Rainfall: 424.55 mm/year
Soil pH: 8.20
Nitrogen: 0.03 mg/kg
Phosphorous: 8.40 mg/kg
Potassium: 8.39 mg/kg


In [8]:
geo_lookup = {
    'tree': tree,
    'coords': coords,
    'values_climate': values_climate,   # columns: [avg_rainfall_mm, soil_ph]
    'values_NPK': values_NPK   # columns: [Nitrogen, Phosphorous, Potassium]
}

joblib.dump(geo_lookup, '../backend/geo_lookup.pkl')
print("Saved geo_lookup.pkl to backend/")

Saved geo_lookup.pkl to backend/


In [10]:
# Quick reload test
loaded = joblib.load('../backend/geo_lookup.pkl')
print("Keys in loaded object:", loaded.keys())
print("KDTree type:", type(loaded['tree']))
print("Coordinates shape in loaded object:", loaded['coords'].shape)
print("Climate values shape in loaded object:", loaded['values_climate'].shape)
print("NPK values shape in loaded object:", loaded['values_NPK'].shape)

Keys in loaded object: dict_keys(['tree', 'coords', 'values_climate', 'values_NPK'])
KDTree type: <class 'scipy.spatial._ckdtree.cKDTree'>
Coordinates shape in loaded object: (1137, 2)
Climate values shape in loaded object: (1137, 2)
NPK values shape in loaded object: (1137, 3)


### Summary

- **KDTree** created for ~1132 GEE grid points.
- Lookup dictionary saved as `backend/geo_lookup.pkl`.
- In the FastAPI backend, load this file and use:
  ```python
  dist, idx = tree.query([[lon, lat]])
  rainfall, ph = values[idx[0]]